# 80/20 XGBoost pipeline smoke test — no saving

Place this notebook in the project root, at the same level as `run_xgboost.sh`.

Expected layout:

```text
new_pipeline_xgb/
  run_xgboost.sh
  src/
    functions.py
    pipeline.py
  data/
    transcriptomics_filtered.joblib
    proteomics_matched.joblib
```

This notebook runs a tiny in-memory 80/20 test and displays the outputs. It does **not** write `.h5` files or modify project files.


In [1]:
# >>> pRNA figure routing (added by the pRNA update; keep as the first code cell)
# Every figure is shown inline. Every file written with savefig (png, pdf, svg) is redirected to
# figures_pdf/<notebook name>/ next to this notebook; sub-folders of the original path are preserved.
import os as _fig_os
from pathlib import Path as _FigPath
import matplotlib as _fig_mpl
import matplotlib.pyplot as _fig_plt
from matplotlib.figure import Figure as _FigFigure

NOTEBOOK_NAME = "new_pipeline_xgb_smoke_test"
FIG_DIR = (_FigPath.cwd() / "figures_pdf" / NOTEBOOK_NAME).resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _route_figure_path(fname):
    p = _FigPath(_fig_os.fspath(fname))
    full = p if p.is_absolute() else (_FigPath.cwd() / p)
    try:
        full.resolve().relative_to(FIG_DIR)
        return fname                                  # already inside this notebook's figure folder
    except ValueError:
        pass
    try:
        rel = _FigPath(_fig_os.path.relpath(full.resolve(), _FigPath.cwd()))
    except ValueError:                                # e.g. another drive on Windows
        rel = _FigPath(p.name)
    parts = [x for x in rel.parts if x not in ("..", ".", "figures_pdf", NOTEBOOK_NAME)]
    target = FIG_DIR.joinpath(*parts) if parts else FIG_DIR / p.name
    target.parent.mkdir(parents=True, exist_ok=True)
    return target


if not getattr(_FigFigure.savefig, "_prna_routed", False):
    _fig_orig_savefig = _FigFigure.savefig

    def _fig_routed_savefig(self, fname, *args, **kwargs):
        if isinstance(fname, (str, bytes, _FigPath)) or hasattr(fname, "__fspath__"):
            fname = _route_figure_path(fname)
        return _fig_orig_savefig(self, fname, *args, **kwargs)

    _fig_routed_savefig._prna_routed = True
    _FigFigure.savefig = _fig_routed_savefig

    _fig_orig_close = _fig_plt.close

    def _fig_close_and_show(fig=None):
        """Display figures that are closed before Jupyter could render them, then close."""
        import inspect as _fig_inspect
        _caller = _fig_inspect.currentframe().f_back
        _caller_mod = (_caller.f_globals.get("__name__", "") if _caller is not None else "")
        if _caller_mod.startswith(("matplotlib", "ipykernel")):   # backend housekeeping, not user code
            return _fig_orig_close(fig)
        if "inline" in str(_fig_mpl.get_backend()).lower():
            from IPython.display import display as _fig_display
            open_nums = set(_fig_plt.get_fignums())
            if fig is None:
                figs = [_fig_plt.gcf()] if open_nums else []
            elif isinstance(fig, str) and fig == "all":
                figs = [_fig_plt.figure(n) for n in sorted(open_nums)]
            elif isinstance(fig, _FigFigure):
                figs = [fig] if getattr(fig, "number", None) in open_nums else []
            elif isinstance(fig, int):
                figs = [_fig_plt.figure(fig)] if fig in open_nums else []
            else:
                figs = []
            for _f in figs:
                if not getattr(_f, "_prna_displayed", False):
                    _f._prna_displayed = True
                    _fig_display(_f)
        return _fig_orig_close(fig)

    _fig_plt.close = _fig_close_and_show

    _fig_orig_show = _fig_plt.show

    def _fig_show_and_mark(*args, **kwargs):
        for _n in _fig_plt.get_fignums():                    # shown now -> never re-displayed on close
            _fig_plt.figure(_n)._prna_displayed = True
        return _fig_orig_show(*args, **kwargs)

    _fig_plt.show = _fig_show_and_mark

print("Figures are saved to", FIG_DIR)

Figures are saved to C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. xgb\new_pipeline_xgb\figures_pdf\new_pipeline_xgb_smoke_test


In [2]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
from joblib import load
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT.parent / "data"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)
print("DATA_DIR:", DATA_DIR)

assert (SRC_DIR / "functions.py").exists(), f"Missing: {SRC_DIR / 'functions.py'}"
assert (SRC_DIR / "pipeline.py").exists(), f"Missing: {SRC_DIR / 'pipeline.py'}"
assert (DATA_DIR / "transcriptomics_filtered.joblib").exists(), f"Missing: {DATA_DIR / 'transcriptomics_filtered.joblib'}"
assert (DATA_DIR / "proteomics_matched.joblib").exists(), f"Missing: {DATA_DIR / 'proteomics_matched.joblib'}"

sys.path.insert(0, str(SRC_DIR))

from functions import train_test_pipeline_for_protein, predict_with_artifact, validate_normalization
from pipeline import build_param_grid, detect_protein_axis, select_proteins

print("Imports OK")

PROJECT_ROOT: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. xgb\new_pipeline_xgb
SRC_DIR: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. xgb\new_pipeline_xgb\src
DATA_DIR: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\03. xgb\data
Imports OK


## Test settings

Defaults here are deliberately tiny so the notebook finishes quickly. Increase these after the smoke test works.


In [3]:
# Tiny smoke-test settings. Adjust if needed.
NORMALIZATIONS = ["none", "qbin50"]   # e.g. ["none", "zscore", "rank", "qbin50"]
PROTEIN_START = 1                       # 1-based index
PROTEIN_END = 1                         # keep 1 protein for a quick notebook test
N_SAMPLES = 30                          # use None for all 369 samples
SELECT_K = 50                           # full run default was 2000
GRID_PRESET = "tiny"                   # full run default was "fast"
MAX_PARAM_COMBOS = 2                    # full run default was 8
SPLITS = 3
EARLY_STOPPING = 5                      # full run default was 10
TEST_FRACTION = 0.20
RANDOM_STATE = 42
XGB_THREADS = 2
USE_GPU = False

for n in NORMALIZATIONS:
    validate_normalization(n)

print("Notebook smoke-test settings:")
for name, value in {
    "NORMALIZATIONS": NORMALIZATIONS,
    "PROTEIN_START": PROTEIN_START,
    "PROTEIN_END": PROTEIN_END,
    "N_SAMPLES": N_SAMPLES,
    "SELECT_K": SELECT_K,
    "GRID_PRESET": GRID_PRESET,
    "MAX_PARAM_COMBOS": MAX_PARAM_COMBOS,
    "SPLITS": SPLITS,
    "EARLY_STOPPING": EARLY_STOPPING,
    "TEST_FRACTION": TEST_FRACTION,
    "XGB_THREADS": XGB_THREADS,
    "USE_GPU": USE_GPU,
}.items():
    print(f"  {name}: {value}")

Notebook smoke-test settings:
  NORMALIZATIONS: ['none', 'qbin50']
  PROTEIN_START: 1
  PROTEIN_END: 1
  N_SAMPLES: 30
  SELECT_K: 50
  GRID_PRESET: tiny
  MAX_PARAM_COMBOS: 2
  SPLITS: 3
  EARLY_STOPPING: 5
  TEST_FRACTION: 0.2
  XGB_THREADS: 2
  USE_GPU: False


## Load data and detect protein orientation

In [4]:
X = load(DATA_DIR / "transcriptomics_filtered.joblib")
Y = load(DATA_DIR / "proteomics_matched.joblib")

X.index.name = "samples"
X.columns.name = "genes"

all_proteins, orientation, index_overlap, column_overlap = detect_protein_axis(X, Y)
proteins_to_run = select_proteins(
    all_proteins,
    protein=None,
    protein_start=PROTEIN_START,
    protein_end=PROTEIN_END,
)

print("Transcriptomics shape:", X.shape)
print("Proteomics shape:", Y.shape)
print("Transcriptomics index examples:", list(map(str, X.index[:3])))
print("Proteomics row/index examples:", list(map(str, Y.index[:3])))
print("Proteomics column examples:", list(map(str, Y.columns[:3])))
print("Detected proteomics orientation:", orientation)
print("Sample overlap using Y rows/index:", index_overlap)
print("Sample overlap using Y columns:", column_overlap)
print("Number of detected proteins:", len(all_proteins))
print("Selected proteins:", proteins_to_run)

Transcriptomics shape: (369, 22614)
Proteomics shape: (369, 5113)
Transcriptomics index examples: ['MDAMB468_BREAST', 'SH4_SKIN', 'AU565_BREAST']
Proteomics row/index examples: ['MDAMB468_BREAST', 'SH4_SKIN', 'AU565_BREAST']
Proteomics column examples: ['A2M', 'AAAS', 'AACS']
Detected proteomics orientation: samples_as_rows__proteins_as_columns
Sample overlap using Y rows/index: 369
Sample overlap using Y columns: 0
Number of detected proteins: 5113
Selected proteins: ['A2M']


## Run the 80/20 test in memory

This calls `train_test_pipeline_for_protein(...)` directly and does not call the H5 save function.


In [5]:
param_grid = build_param_grid(use_gpu=USE_GPU, preset=GRID_PRESET)
results = []
artifacts = {}

for protein in proteins_to_run:
    for normalization in NORMALIZATIONS:
        print("=" * 100)
        print(f"Protein: {protein} | normalization: {normalization}")
        result = train_test_pipeline_for_protein(
            X=X,
            y=Y,
            protein_name=protein,
            param_grid=param_grid,
            normalization=normalization,
            test_fraction=TEST_FRACTION,
            random_state=RANDOM_STATE,
            n_splits=SPLITS,
            select_k=SELECT_K,
            early_stopping_rounds=EARLY_STOPPING,
            max_param_combos=MAX_PARAM_COMBOS,
            n_samples=N_SAMPLES,
            use_gpu=USE_GPU,
            xgb_threads=XGB_THREADS,
        )
        summary = result["summary"]
        artifact = result["model_artifact"]
        artifacts[(protein, normalization)] = artifact
        results.append({
            "protein": summary["protein"],
            "normalization": normalization,  # displayed only; not stored by the minimal H5 pipeline
            "test_mse": summary["test_mse"],
            "test_pearson_corr": summary["test_pearson_corr"],
            "test_cosine_similarity": summary["test_cosine_similarity"],
            "n_train_samples": len(summary["train_samples"]),
            "n_test_samples": len(summary["test_samples"]),
            "n_selected_features": len(summary["selected_features"]),
            "best_params": summary["best_params"],
            "model_object": artifact["model"],
        })

summary_df = pd.DataFrame(results)
summary_df = summary_df.sort_values(["protein", "test_pearson_corr"], ascending=[True, False], na_position="last")
display(summary_df)

Protein: A2M | normalization: none
Protein: A2M | normalization: qbin50


,protein,normalization,test_mse,test_pearson_corr,test_cosine_similarity,n_train_samples,n_test_samples,n_selected_features,best_params,model_object
0,A2M,none,0.691805,0.303232,0.33911,24,6,50,"{'max_depth': 3, 'min_child_weight': 1, 'learn...","XGBRegressor(base_score=None, booster=None, ca..."
1,A2M,qbin50,0.691805,0.303232,0.33911,24,6,50,"{'max_depth': 3, 'min_child_weight': 1, 'learn...","XGBRegressor(base_score=None, booster=None, ca..."


## Inspect the best in-memory model

In [6]:
if len(summary_df) == 0:
    raise RuntimeError("No results were produced.")

best_row = summary_df.iloc[0]
best_key = (best_row["protein"], best_row["normalization"])
best_artifact = artifacts[best_key]

print("Best row by Pearson among this notebook run:")
display(pd.DataFrame([best_row.drop(labels=["model_object"]).to_dict()]))

print("Best parameters:")
print(json.dumps(best_artifact["params"], indent=2, sort_keys=True))

print("Model artifact fields:")
print(sorted(best_artifact.keys()))

print("First 20 selected features:")
print(best_artifact["selected_features"][:20])

Best row by Pearson among this notebook run:


,protein,normalization,test_mse,test_pearson_corr,test_cosine_similarity,n_train_samples,n_test_samples,n_selected_features,best_params
0,A2M,none,0.691805,0.303232,0.33911,24,6,50,"{'max_depth': 3, 'min_child_weight': 1, 'learn..."


Best parameters:
{
  "colsample_bytree": 0.8,
  "device": "cpu",
  "gamma": 0.0,
  "learning_rate": 0.05,
  "max_bin": 256,
  "max_depth": 3,
  "min_child_weight": 1,
  "n_estimators": 27,
  "reg_alpha": 0.1,
  "reg_lambda": 1.0,
  "subsample": 0.9,
  "tree_method": "hist"
}
Model artifact fields:
['artifact_kind', 'feature_importance_gain', 'feature_selector', 'model', 'n_training_samples', 'normalizer', 'params', 'params_with_cv_metadata', 'protein', 'selected_features', 'training_samples', 'use_gpu', 'xgb_threads', 'xgboost_version']
First 20 selected features:
['FAM83A-AS1', 'IPCEF1', 'ANKDD1A', 'RSPH3', 'AL117332.1', 'FAR2', 'LANCL3', 'AL357033.4', 'TINAGL1', 'MALL', 'RASAL1', 'ATP2A3', 'PSMD5', 'DOCK3', 'GJB4', 'LCN2', 'ESYT2', 'FAM83A', 'GPRIN2', 'AC108676.1']
